# Phase B-1〜B-5: 路線価予測モデル

LightGBM + Spatial CV による路線価予測モデルの構築・評価・解釈。  
**合成データで動作確認済み。`DATA_PATH` を変えるだけで実データに適用可能。**

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

try:
    import japanize_matplotlib
except ImportError:
    pass

from rex_io import load_and_prepare
from features import build_feature_matrix, FEATURE_COLS
from train import spatial_cv, train, feature_importance, evaluate, compute_shap, HAS_SHAP
from residual_check import add_residuals, full_diagnosis, residual_by_chikukbn, residual_by_prefecture

plt.rcParams["figure.dpi"] = 120

# ── データパス（ここだけ変えれば実データに切り替え可能）──────────────
DATA_PATH = Path("/Users/KASU/REX_data_2020-2022/2022/nouhin_line_2022.shp")    
# DATA_PATH = Path("/実データパス/nouhin_line_2022.shp")

## 1. データ読み込みと特徴量生成

In [ ]:
gdf = load_and_prepare(DATA_PATH)
print(f"件数: {len(gdf):,}")

X, y, groups = build_feature_matrix(gdf, use_fast=True)
print(f"特徴量数: {X.shape[1]}")
print(f"グループ（都道府県）数: {groups.nunique()}")
X.describe().T[["mean", "std", "min", "max"]]

## 2. 目的変数の分布確認

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左: 元の路線価（右裾が長い）
axes[0].hist(gdf["kakaku"], bins=60, color="steelblue", edgecolor="white")
axes[0].set_xlabel("路線価（千円/㎡）")
axes[0].set_title("路線価の分布（元スケール）")

# 右: 対数変換後（正規分布に近い）→ 目的変数として使用
axes[1].hist(y, bins=60, color="coral", edgecolor="white")
axes[1].set_xlabel("log(路線価 + 1)")
axes[1].set_title("路線価の分布（対数変換後）← 目的変数")

plt.tight_layout()
plt.show()

print(f"歪度（元）: {gdf['kakaku'].skew():.2f}  →  歪度（log変換後）: {y.skew():.2f}")

## 3. Spatial CV（空間交差検証）

都道府県コードをグループとした GroupKFold。  
空間的に隣接するデータがトレーニングとテストに同時に入らないようにする。

In [ ]:
cv_result = spatial_cv(X, y, groups, n_splits=5)
cv_result

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
folds = cv_result[cv_result["fold"].apply(lambda x: str(x).isdigit() if isinstance(x, str) else isinstance(x, (int, float)) and x == int(x))].copy()
folds = folds[folds["fold"].astype(str).str.match(r"^\d+$")]
folds["fold"] = folds["fold"].astype(int)

axes[0].bar(folds["fold"], folds["r2_log"], color="steelblue", edgecolor="white")
axes[0].axhline(cv_result[cv_result["fold"] == "mean"]["r2_log"].values[0],
                color="red", linestyle="--", label="平均")
axes[0].set_ylim(0.9, 1.0)
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("R²（log空間）")
axes[0].set_title("Spatial CV: R² per fold")
axes[0].legend()

axes[1].bar(folds["fold"], folds["rmse_real"], color="coral", edgecolor="white")
axes[1].axhline(cv_result[cv_result["fold"] == "mean"]["rmse_real"].values[0],
                color="red", linestyle="--", label="平均")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("RMSE（千円/㎡）")
axes[1].set_title("Spatial CV: RMSE per fold")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. フルデータ学習

In [ ]:
model = train(X, y)
metrics = evaluate(model, X, y)

print("=== 学習データ上の評価指標 ===")
for k, v in metrics.items():
    print(f"  {k:15s}: {v}")

## 5. 特徴量重要度

In [ ]:
imp = feature_importance(model, X, importance_type="gain")

fig, ax = plt.subplots(figsize=(8, 7))
imp_plot = imp.head(15)
colors = ["#2166ac" if "lag" in i or "pre" in i
          else "#d6604d" if "chiku" in i or "commercial" in i or "swari" in i
          else "#4dac26"
          for i in imp_plot.index]
ax.barh(imp_plot.index[::-1], imp_plot.values[::-1], color=colors[::-1])
ax.set_xlabel("重要度（gain）")
ax.set_title("特徴量重要度 Top 15")

from matplotlib.patches import Patch
legend = [
    Patch(color="#2166ac", label="空間ラグ・前年価格"),
    Patch(color="#d6604d", label="属性（地区・借地権）"),
    Patch(color="#4dac26", label="ジオメトリ・位置"),
]
ax.legend(handles=legend, loc="lower right")
plt.tight_layout()
plt.show()

## 6. SHAP による解釈

各特徴量が個々の路線価予測にどう寄与しているかを可視化する。

In [ ]:
if HAS_SHAP:
    import shap
    shap_values, explainer, X_sample = compute_shap(model, X, sample_n=300)

    # Summary plot（全特徴量のSHAP分布）
    plt.figure()
    shap.summary_plot(shap_values, X_sample, show=False)
    plt.title("SHAP Summary Plot")
    plt.tight_layout()
    plt.show()
else:
    print("shap が未インストールです: pip install shap")

In [ ]:
if HAS_SHAP:
    # Bar plot（平均|SHAP|による重要度）
    plt.figure()
    shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
    plt.title("SHAP Feature Importance（平均 |SHAP|）")
    plt.tight_layout()
    plt.show()

## 7. 残差診断

モデルに取り残された空間構造がないかを確認する。  
残差のMoran's Iが有意な場合はモデル改善（空間誤差モデルとの比較等）が必要。

In [ ]:
y_pred = model.predict(X)
gdf_res = add_residuals(gdf, y, y_pred)

from residual_check import plot_residuals, residual_moran
plot_residuals(gdf_res)

In [ ]:
print("=== 残差のMoran's I（空間的自己相関の残存チェック）===")
moran_res = residual_moran(gdf_res)
for k, v in moran_res.items():
    print(f"  {k}: {v}")

In [ ]:
print("=== 地区区分別の残差（abs_error 中央値）===")
by_chiku = residual_by_chikukbn(gdf_res)

fig, ax = plt.subplots(figsize=(9, 4))
medians = by_chiku["abs_error"]["median"].sort_values(ascending=False)
colors = ["#d73027" if v > medians.median() else "#4575b4" for v in medians]
ax.barh(medians.index[::-1], medians.values[::-1], color=colors[::-1])
ax.set_xlabel("abs_error の中央値（log空間）")
ax.set_title("地区区分別 予測誤差")
plt.tight_layout()
plt.show()

## 8. モデルの保存

実データ適用後、最終モデルを保存する。

In [ ]:
import pickle
from pathlib import Path

out_dir = Path("../outputs")
out_dir.mkdir(exist_ok=True)

# モデル保存
model_path = out_dir / "lgbm_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f)

# CV結果・特徴量重要度を保存
cv_result.to_csv(out_dir / "cv_results.csv", index=False)
imp.to_csv(out_dir / "feature_importance.csv")

print(f"モデル保存: {model_path}")
print(f"CV結果保存: {out_dir / 'cv_results.csv'}")
print(f"重要度保存: {out_dir / 'feature_importance.csv'}")